# Notebook 3 — AI Review Engine

**What you'll learn:**
1. How prompt engineering affects review quality
2. How to call the OpenAI API with structured output
3. How to parse and display the AI's response
4. End-to-end: review a real diff

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

## Why Prompt Engineering Matters

Generic prompt → generic output. Specific prompt → actionable output.

Key techniques used in this project:

| Technique | What it does |
|---|---|
| Role assignment | "You are a senior software engineer..." sets the model's persona |
| Structured output | Force JSON so we can parse it programmatically |
| Low temperature (0.3) | More deterministic, less creative hallucination |
| Negative instructions | "No generic advice" stops filler output |
| Severity categories | Lets us prioritise and filter issues |

## A Deliberately Bad Diff

Let's craft a diff with several real issues for the AI to find.

In [ ]:
VULNERABLE_DIFF = '''
### File: auth.py (modified)
@@ -0,0 +1,30 @@
+import sqlite3
+import os
+
+DATABASE_PASSWORD = "super_secret_123"  # TODO: move to env
+
+def login(username, password):
+    conn = sqlite3.connect("users.db")
+    cursor = conn.cursor()
+    # Direct string interpolation — SQL injection!
+    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
+    cursor.execute(query)
+    user = cursor.fetchone()
+    conn.close()
+    return user
+
+def get_all_users():
+    conn = sqlite3.connect("users.db")
+    cursor = conn.cursor()
+    results = []
+    all_ids = cursor.execute("SELECT id FROM users").fetchall()
+    for row in all_ids:                      # N+1 query
+        user = cursor.execute(f"SELECT * FROM users WHERE id={row[0]}").fetchone()
+        results.append(user)
+    return results

### File: api.py (modified)
@@ -0,0 +1,12 @@
+import requests
+
+def fetch_user_data(user_id):
+    # No timeout, no error handling
+    response = requests.get(f"https://api.example.com/users/{user_id}")
+    return response.json()
'''
print('Diff ready — contains SQL injection, hardcoded secret, N+1 query, missing error handling.')

## Call the AI Reviewer

In [ ]:
from app.ai_reviewer import AIReviewer

reviewer = AIReviewer()

review = reviewer.review_diff(
    diff=VULNERABLE_DIFF,
    pr_title='Add user authentication and API client',
    pr_description='Implements login and user data fetching.'
)

print('Score:', review.get('score'), '/ 10')
print('Issues found:', len(review.get('issues', [])))
print('\nSummary:')
print(review.get('summary'))

## Inspect Each Issue

In [ ]:
severity_order = {'critical': 0, 'high': 1, 'medium': 2, 'low': 3}
issues = sorted(review.get('issues', []), key=lambda x: severity_order.get(x.get('severity','low'), 4))

for i, issue in enumerate(issues, 1):
    sev = issue.get('severity', 'unknown').upper()
    print(f"\n{'='*60}")
    print(f"Issue #{i}  [{sev}]  {issue.get('issue')}")
    print(f"File     : {issue.get('file')} — line {issue.get('line_hint')}")
    print(f"Why      : {issue.get('explanation')}")
    print(f"Fix      : {issue.get('suggestion')}")

## Generate the GitHub Markdown Comment

In [ ]:
from app.ai_reviewer import AIReviewer
from IPython.display import Markdown, display

markdown = AIReviewer.format_as_markdown(review)
display(Markdown(markdown))

## Explain a Specific Issue In Depth

In [ ]:
if issues:
    top_issue = issues[0]  # The most critical one
    code_snippet = """
query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
cursor.execute(query)
"""
    explanation = reviewer.explain_issue(top_issue, code_snippet)
    display(Markdown(explanation))

## Test Against a Clean Diff

What happens when the code is actually good?

In [ ]:
CLEAN_DIFF = '''
### File: auth.py (modified)
@@ -0,0 +1,15 @@
+import sqlite3
+import os
+
+def login(username: str, password: str) -> dict | None:
+    conn = sqlite3.connect(os.getenv('DB_PATH', 'users.db'))
+    try:
+        cursor = conn.cursor()
+        # Parameterised query — safe from SQL injection
+        cursor.execute(
+            'SELECT id, username FROM users WHERE username = ? AND password_hash = ?',
+            (username, hash_password(password)),
+        )
+        row = cursor.fetchone()
+        return {'id': row[0], 'username': row[1]} if row else None
+    finally:
+        conn.close()
'''

clean_review = reviewer.review_diff(CLEAN_DIFF, pr_title='Secure the login function')
print('Score:', clean_review.get('score'), '/ 10')
print('Issues:', len(clean_review.get('issues', [])))
print('Summary:', clean_review.get('summary'))

---
**Next:** Open `04_dashboard.ipynb` to visualise review results.